In [1]:
#Importing libraries
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import norm
import zipfile
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
#Loading data
bom=pd.read_csv('zippedData/bom.movie_gross.csv.gz')
bom

,title,studio,domestic_gross,foreign_gross,year
0,Toy Story 3,BV,415000000.0,652000000,2010
1,Alice in Wonderland (2010),BV,334200000.0,691300000,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000,2010
3,Inception,WB,292600000.0,535700000,2010
4,Shrek Forever After,P/DW,238700000.0,513900000,2010
...,...,...,...,...,...
3382,The Quake,Magn.,6200.0,NaN,2018
3383,Edward II (2018 re-release),FM,4800.0,NaN,2018
3384,El Pacto,Sony,2500.0,NaN,2018
3385,The Swan,Synergetic,2400.0,NaN,2018


In [3]:
zip_path = "zippedData/im.db.zip"   # <-- your zip file
extract_path = "zippedData/" 

In [4]:
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
        print("Unzipped successfully!")
else:
    print("Zip file not found!")

Unzipped successfully!


In [5]:
conn = sqlite3.connect("zippedData/im.db")
cur = conn.cursor()

tables=pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)

tables


,name
0,movie_basics
1,directors
2,known_for
3,movie_akas
4,movie_ratings
5,persons
6,principals
7,writers


In [6]:
#Using sqlite3 with pandas
#df=pd.read_sql(query,conn)
#Using sqlite3 with pandas
#df=pd.read_sql(query,conn)
movie_basics=pd.read_sql('SELECT * FROM movie_basics',conn)
movie_basics.head()


,movie_id,primary_title,original_title,start_year,runtime_minutes,genres
0,tt0063540,Sunghursh,Sunghursh,2013,175.0,"Action,Crime,Drama"
1,tt0066787,One Day Before the Rainy Season,Ashad Ka Ek Din,2019,114.0,"Biography,Drama"
2,tt0069049,The Other Side of the Wind,The Other Side of the Wind,2018,122.0,Drama
3,tt0069204,Sabse Bada Sukh,Sabse Bada Sukh,2018,NaN,"Comedy,Drama"
4,tt0100275,The Wandering Soap Opera,La Telenovela Errante,2017,80.0,"Comedy,Drama,Fantasy"


In [7]:
#Using sqlite3 with pandas
#df=pd.read_sql(query,conn)
movie_ratings=pd.read_sql('SELECT * FROM movie_ratings',conn)
movie_ratings.head()

,movie_id,averagerating,numvotes
0,tt10356526,8.3,31
1,tt10384606,8.9,559
2,tt1042974,6.4,20
3,tt1043726,4.2,50352
4,tt1060240,6.5,21


In [8]:
tn=pd.read_csv('zippedData/tn.movie_budgets.csv.gz')
tn

,id,release_date,movie,production_budget,domestic_gross,worldwide_gross
0,1,"Dec 18, 2009",Avatar,"$425,000,000","$760,507,625","$2,776,345,279"
1,2,"May 20, 2011",Pirates of the Caribbean: On Stranger Tides,"$410,600,000","$241,063,875","$1,045,663,875"
2,3,"Jun 7, 2019",Dark Phoenix,"$350,000,000","$42,762,350","$149,762,350"
3,4,"May 1, 2015",Avengers: Age of Ultron,"$330,600,000","$459,005,868","$1,403,013,963"
4,5,"Dec 15, 2017",Star Wars Ep. VIII: The Last Jedi,"$317,000,000","$620,181,382","$1,316,721,747"
...,...,...,...,...,...,...
5777,78,"Dec 31, 2018",Red 11,"$7,000",$0,$0
5778,79,"Apr 2, 1999",Following,"$6,000","$48,482","$240,495"
5779,80,"Jul 13, 2005",Return to the Land of Wonders,"$5,000","$1,338","$1,338"
5780,81,"Sep 29, 2015",A Plague So Pleasant,"$1,400",$0,$0


#4.1 DATA CLEANING

In [9]:
#Data inspecting 
def inspect(df, name):
    print(f"\n--- {name} ---")
    print(df.head())
    print(df.info())
    print(df.describe(include='all'))

inspect(bom, 'BOM Movie Gross')
inspect(tn, 'TN Movie Budgets')
inspect(tables, 'IMDB Movies')


--- BOM Movie Gross ---
                                         title studio  domestic_gross  \
0                                  Toy Story 3     BV     415000000.0   
1                   Alice in Wonderland (2010)     BV     334200000.0   
2  Harry Potter and the Deathly Hallows Part 1     WB     296000000.0   
3                                    Inception     WB     292600000.0   
4                          Shrek Forever After   P/DW     238700000.0   

  foreign_gross  year  
0     652000000  2010  
1     691300000  2010  
2     664300000  2010  
3     535700000  2010  
4     513900000  2010  


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3387 entries, 0 to 3386
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           3387 non-null   object 
 1   studio          3382 non-null   object 
 2   domestic_gross  3359 non-null   float64
 3   foreign_gross   2037 non-null   object 
 4   year            3387 non-null   int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 132.4+ KB
None
            title studio  domestic_gross foreign_gross         year
count        3387   3382    3.359000e+03          2037  3387.000000
unique       3386    257             NaN          1204          NaN
top     Bluebeard    IFC             NaN       1200000          NaN
freq            2    166             NaN            23          NaN
mean          NaN    NaN    2.874585e+07           NaN  2013.958075
std           NaN    NaN    6.698250e+07           NaN     2.478141
min           NaN    NaN    1.0000

In [10]:
#Find Missing Values
print("BOM Movie Gross missing values:")
print(bom.isnull().sum())

print("\nTN Movie Budgets missing values:")
print(tn.isnull().sum())

print("\nIMDB Movies (tables) missing values:")
print(tables.isnull().sum())

BOM Movie Gross missing values:
title                0
studio               5
domestic_gross      28
foreign_gross     1350
year                 0
dtype: int64

TN Movie Budgets missing values:
id                   0
release_date         0
movie                0
production_budget    0
domestic_gross       0
worldwide_gross      0
dtype: int64

IMDB Movies (tables) missing values:
name    0
dtype: int64


In [11]:
# Establishing the data types of foreign gross
bom['foreign_gross'].dtype

dtype('O')

In [12]:
# # Establishing the data types of domestic gross
bom['domestic_gross'].dtype

dtype('float64')

In [13]:
#Establishing the data types of domestic gross
bom['studio'].dtype

dtype('O')

In [14]:
#Converting foreign_gross
bom['foreign_gross'] = pd.to_numeric(bom['foreign_gross'], errors='coerce')


In [15]:
#Converting Date into Datetime
#For bom.movie_gross data, we are using the year column
# Convert Year to integer
bom['year'] = bom['year'].astype(int)

#For tn.movie_budgets, we format the release_date column
tn['release_date'] = pd.to_datetime(tn['release_date'])

#Extract Month and Year
tn['release_month'] = tn['release_date'].dt.month
tn['release_year'] = tn['release_date'].dt.year

In [16]:
# Clean TN Movie Budgets 
# Remove $ and commas
cols = ['production_budget', 'domestic_gross', 'worldwide_gross']

for col in cols:
    tn[col] = tn[col].replace({'\$': '', ',': ''}, regex=True)
    tn[col] = pd.to_numeric(tn[col], errors='coerce')

In [17]:
# Clean Movie_basics  Data
movie_basics.columns

Index(['movie_id', 'primary_title', 'original_title', 'start_year',
       'runtime_minutes', 'genres'],
      dtype='object')

In [18]:

# Keep only relevant columns

movie_basics = movie_basics[['movie_id', 'primary_title', 'original_title', 'start_year',
       'runtime_minutes', 'genres']]

# Rename for merging
movie_basics.rename(columns={'primary_title': 'title'}, inplace=True)

In [19]:
#checking columns in movie_ratings data
movie_ratings.columns

Index(['movie_id', 'averagerating', 'numvotes'], dtype='object')

In [20]:
# Cleaning Movie_ratings data
# Merge ratings with movie_basics
ratings_basics = pd.merge(movie_basics, movie_ratings, on='movie_id', how='inner')

In [21]:
# Lowercase titles for consistency
bom['title'] = bom['title'].str.lower()
tn['movie'] = tn['movie'].str.lower()
ratings_basics['title'] = ratings_basics['title'].str.lower()

In [23]:
# Replacing Missing Values
# Replacing studio values
bom['studio']=bom['studio'].fillna('unknown')
#Replacing domestic_grosss missing values
domestic_median=bom['domestic_gross'].median()
bom['domestic_gross'] = bom['domestic_gross'].fillna(domestic_median)

#Replacing foreign_grosss missing values
foreign_mean = bom['foreign_gross'].mean()
bom['foreign_gross'] = bom['foreign_gross'].fillna(foreign_mean)

In [24]:
# Merging Datasets
# Merge revenue + budget
merged = pd.merge(
    bom,
    tn,
    left_on='title',
    right_on='movie',
    how='inner'
)

In [25]:
# Merge with ratings
merged_1 = pd.merge(
    merged,
    ratings_basics,
    on='title',
    how='inner'
)

# Total revenue (biggest success metric)

In [ ]:
# Create total gross:
bom['total_gross'] = bom['domestic_gross'] + bom['foreign_gross']

In [ ]:
# Top-performing movies
# Focus on high-revenue genres
bom.sort_values('total_gross', ascending=False).head(10)

,title,studio,domestic_gross,foreign_gross,year,total_gross
727,Marvel's The Avengers,BV,623400000.0,895500000.0,2012,1.518900e+09
1875,Avengers: Age of Ultron,BV,459000000.0,946400000.0,2015,1.405400e+09
3080,Black Panther,BV,700100000.0,646900000.0,2018,1.347000e+09
328,Harry Potter and the Deathly Hallows Part 2,WB,381000000.0,960500000.0,2011,1.341500e+09
2758,Star Wars: The Last Jedi,BV,620200000.0,712400000.0,2017,1.332600e+09
3081,Jurassic World: Fallen Kingdom,Uni.,417700000.0,891800000.0,2018,1.309500e+09
1127,Frozen,BV,400700000.0,875700000.0,2013,1.276400e+09
2759,Beauty and the Beast (2017),BV,504000000.0,759500000.0,2017,1.263500e+09
3082,Incredibles 2,BV,608600000.0,634200000.0,2018,1.242800e+09
1128,Iron Man 3,BV,409000000.0,805800000.0,2013,1.214800e+09


In [ ]:
# Studio performance
bom.groupby('studio')['total_gross'].mean().sort_values(ascending=False)

studio
HC              8.703000e+08
P/DW            5.076500e+08
BV              4.199350e+08
GrtIndia        2.542000e+08
WB (NL)         2.313279e+08
                    ...     
FOAK            1.243000e+05
IVP             1.121000e+05
Darin Southa    9.840000e+04
ITL             5.290000e+04
WOW             4.940000e+04
Name: total_gross, Length: 258, dtype: float64

In [ ]:
# Yearly trends
bom.groupby('year')['total_gross'].mean()

# Is revenue growing over tim

year
2010    7.843205e+07
2011    8.458316e+07
2012    9.843568e+07
2013    1.098049e+08
2014    9.926239e+07
2015    1.021090e+08
2016    1.113479e+08
2017    1.302840e+08
2018    1.258878e+08
Name: total_gross, dtype: float64

In [ ]:
# Domestic vs Foreign market
# Global appeal is critical
# Think globally, not locally
bom[['domestic_gross', 'foreign_gross']].mean()

domestic_gross    2.874585e+07
foreign_gross     7.505704e+07
dtype: float64

In [ ]:
# Budget vs revenue (if merged with tn dataset)
# Budget strategically
# High budget ≠ guaranteed success
# ROI matters more than revenue
merged_1['profit'] = merged_1['worldwide_gross'] - merged_1['production_budget']
# ROI
merged_1['roi'] = merged_1['profit'] / merged_1['production_budget']

In [ ]:
# Learn from top studios
# Genre choices, Release timing and Franchise strategy

# “To maximize box office success, the studio should prioritize high-budget action and adventure films with global appeal. These genres consistently generate the highest revenues, especially in international markets. Additionally, investing in franchise-based content and focusing on return on investment rather than just total revenue will improve long-term profitability.”